In [ ]:
!pip install -q rasterio pystac rio-tiler

In [ ]:
import os
os.environ.update({
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.TIF,.json",
    "GDAL_HTTP_MULTIPLEX": "YES",
    "GDAL_HTTP_VERSION": "2",
    "VSI_CACHE": "TRUE",
    "VSI_CACHE_SIZE": "200000000",
    "GDAL_CACHEMAX": "512",
    "AWS_NO_SIGN_REQUEST": "YES",
})

In [ ]:
import requests
from urllib.parse import urljoin
import rasterio
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
COLL = ("https://maxar-opendata.s3.amazonaws.com/events/NWT-Canada-Aug-23/"
        "ard/acquisition_collections/104001008A8DAA00_collection.json")

coll_json = requests.get(COLL, timeout=60).json()
item_links = [urljoin(COLL, l["href"]) for l in coll_json["links"] if l["rel"] == "item"]
print(f"{len(item_links)} item links resolved")

In [ ]:
visual_urls = []
for link in item_links:
    try:
        response = requests.get(link, timeout=60)
        response.raise_for_status()
        item_json = response.json()
        rel_href = item_json["assets"]["visual"]["href"]
        abs_href = urljoin(link, rel_href)   # resolve relative to the ITEM's own url
        visual_urls.append(abs_href)
    except Exception as e:
        print(f"Warning: Failed to fetch or parse {link}. Error: {e}")

if visual_urls:
    print(f"Successfully resolved {len(visual_urls)} visual URLs.")
    print("First URL:", visual_urls[0])
else:
    print("No visual URLs could be resolved.")

In [ ]:
thumb_size = 256
images = []

for i, url in enumerate(visual_urls):
    try:
        with rasterio.open(url) as src:
            arr = src.read(out_shape=(src.count, thumb_size, thumb_size))
            arr = np.moveaxis(arr, 0, -1)
        images.append((url.split("/")[-1], arr))
        print(f"[{i+1}/{len(visual_urls)}] read {url.split('/')[-1]}")
    except Exception as e:
        print(f"[{i+1}/{len(visual_urls)}] FAILED: {e}")
        images.append((url.split("/")[-1], None))

In [ ]:
n = len(images)
cols = 5
rows = -(-n // cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten()

for ax, (name, arr) in zip(axes, images):
    if arr is not None:
        ax.imshow(arr)
    ax.set_title(name[:20], fontsize=8)
    ax.axis("off")

for ax in axes[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Image Preprocessing Tips

Here are some common image preprocessing techniques you can apply to your `images` data:

1.  **Adjusting Brightness and Contrast**: This can enhance features and make details more visible. Brightness can be adjusted by adding a constant value to all pixel values, while contrast can be adjusted by multiplying pixel values by a factor.

2.  **Remapping Pixel Values (Normalization/Standardization)**: Often, images are scaled to a specific range (e.g., 0-1 or 0-255). This can be crucial for algorithms that expect input in a certain range or to ensure consistent input across different images.

3.  **Thresholding**: This technique converts a grayscale image into a binary image (black and white) by setting a threshold value. Pixels above the threshold become one value (e.g., white), and those below become another (e.g., black). This is useful for segmenting objects from the background.

4.  **Removing Pixels Below a Certain Amount**: This can be a form of noise reduction or a specific type of thresholding where pixels with values below a certain intensity are set to zero (black). This can help to remove dark artifacts or very faint features.

5.  **Color Space Conversion**: Depending on your task, you might want to convert images from RGB to grayscale, HSV, or other color spaces.

6.  **Filtering (e.g., Gaussian Blur, Median Filter)**: Filters can be used for smoothing images (reducing noise) or sharpening them. Gaussian blur is often used to reduce noise and detail, while median filters are effective at removing salt-and-pepper noise.

7.  **Resizing/Cropping**: Standardizing image dimensions is often necessary for batch processing or model input.

Let's try adjusting brightness and contrast as an example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cleaned: Redundant download logic removed as 'images' is already loaded in the environment.

def adjust_brightness_contrast(image, brightness_factor, contrast_factor):
    # Ensure image is float type for calculations
    img_float = image.astype(np.float32)

    # Adjust contrast
    img_float = img_float * contrast_factor

    # Adjust brightness
    img_float = img_float + brightness_factor

    # Clip pixel values to valid range (0-255 for 8-bit images)
    img_clipped = np.clip(img_float, 0, 255)

    return img_clipped.astype(np.uint8)

# Example usage: Adjust brightness and contrast for the first loaded image
example_image_data = None
example_image_name = ""
for name, arr in images:
    if arr is not None:
        example_image_data = arr
        example_image_name = name
        break

if example_image_data is not None:
    # Define adjustment factors
    brightness = -100  # Decrease brightness significantly
    contrast = 1.5     # Increase contrast

    adjusted_image = adjust_brightness_contrast(example_image_data, brightness, contrast)

    # Display original and adjusted images
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(example_image_data)
    axes[0].set_title(f"Original: {example_image_name[:20]}")
    axes[0].axis("off")

    axes[1].imshow(adjusted_image)
    axes[1].set_title(f"Adjusted (B:{brightness}, C:{contrast}): {example_image_name[:20]}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No valid images found in memory to demonstrate adjustment.")

### Applying Brightness and Contrast to All Images

Now, let's apply the same brightness and contrast adjustment to all the loaded images and display them to see the effect.

In [ ]:
adjusted_images = []

brightness_factor = -100  # Example: decrease brightness significantly
contrast_factor = 1.5    # Example: increase contrast

for name, original_arr in images:
    if original_arr is not None:
        adjusted_arr = adjust_brightness_contrast(original_arr, brightness_factor, contrast_factor)
        adjusted_images.append((name, original_arr, adjusted_arr))

# Display all original and adjusted images in a grid
n_valid_images = len(adjusted_images)
rows = n_valid_images

fig, axes = plt.subplots(rows, 2, figsize=(10, rows * 5))

if rows == 1:
    # If only one image, axes is a 1D array of length 2
    ax_orig = axes[0]
    ax_adj = axes[1]
    name, original_arr, adjusted_arr = adjusted_images[0]

    ax_orig.imshow(original_arr)
    ax_orig.set_title(f"Original: {name[:20]}")
    ax_orig.axis("off")

    ax_adj.imshow(adjusted_arr)
    ax_adj.set_title(f"Adjusted (B:{brightness_factor}, C:{contrast_factor}): {name[:20]}")
    ax_adj.axis("off")
else:
    # For multiple images, axes is a 2D array
    for i, (name, original_arr, adjusted_arr) in enumerate(adjusted_images):
        ax_orig = axes[i, 0]
        ax_adj = axes[i, 1]

        ax_orig.imshow(original_arr)
        ax_orig.set_title(f"Original: {name[:20]}")
        ax_orig.axis("off")

        ax_adj.imshow(adjusted_arr)
        ax_adj.set_title(f"Adjusted (B:{brightness_factor}, C:{contrast_factor}): {name[:20]}")
        ax_adj.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

def is_not_black_tile(image_array, threshold=5.0):
    """
    Checks if an image tile contains actual content or is a black/empty border tile.

    Parameters:
    - image_array: numpy array representing the image (H, W, C) or (H, W)
    - threshold: minimum average pixel intensity across all channels to be considered valid

    Returns:
    - True if the tile contains real imagery, False if it is a black/empty tile.
    """
    if image_array is None:
        return False

    # Calculate the average intensity across all pixels and channels
    avg_intensity = np.mean(image_array)
    return avg_intensity > threshold

# Quick test on our loaded images
valid_tiles = []
for name, arr in images:
    if arr is not None and is_not_black_tile(arr):
        valid_tiles.append((name, arr))

print(f"Total Loaded Tiles: {len(images)}")
print(f"Valid Non-Black Tiles: {len(valid_tiles)}")

### Saving Adjusted Images to a Folder

Let's create a new directory and save the adjusted images to it. This will allow you to store and access these processed versions separately.

In [ ]:
# Assuming thumb_size is 256 from previous cells
# We have 25 images, so a 5x5 grid makes sense (5 rows, 5 columns)
import os
import numpy as np
import matplotlib.pyplot as plt

rows = 5
cols = 5
image_width = thumb_size
image_height = thumb_size

# Create an empty canvas for the composite image
# Assuming images are 3-channel (RGB)
composite_image = np.zeros((rows * image_height, cols * image_width, 3), dtype=np.uint8)

# Populate the composite image with adjusted images
for i, (name, original_arr, adjusted_arr) in enumerate(adjusted_images):
    if adjusted_arr is not None:
        row_idx = i // cols
        col_idx = i % cols

        start_y = row_idx * image_height
        end_y = (row_idx + 1) * image_height
        start_x = col_idx * image_width
        end_x = (col_idx + 1) * image_width

        composite_image[start_y:end_y, start_x:end_x] = adjusted_arr

# Display the final composite image
plt.figure(figsize=(cols * 2.5, rows * 2.5)) # Adjust figure size based on grid dimensions
plt.imshow(composite_image)
plt.title("Composite Adjusted Storm Image")
plt.axis("off")
plt.tight_layout()
plt.show()

# Define output folder here to avoid NameError
output_folder = "adjusted_images_output"
os.makedirs(output_folder, exist_ok=True)

# Save the composite image
composite_output_path = os.path.join(output_folder, "composite_adjusted_storm.png")
plt.imsave(composite_output_path, composite_image)
print(f"Composite image saved to: {composite_output_path}")

In [ ]:
# Removed: Duplicate save block. The composite storm image is already created and saved in the previous step.

In [ ]:
import os

output_folder = "adjusted_images_output"
os.makedirs(output_folder, exist_ok=True)

print(f"Saving {len(adjusted_images)} adjusted images to '{output_folder}'...")

for name, _, adjusted_arr in adjusted_images:
    # Change the extension to .png for successful saving with plt.imsave
    output_filename = os.path.splitext(name)[0] + '.png'
    output_path = os.path.join(output_folder, f"adjusted_{output_filename}")

    # Matplotlib's imsave expects a 0-255 range for uint8 or 0-1 for float.
    # Since our adjust_brightness_contrast function already outputs uint8 in 0-255,
    # we can directly use it.
    plt.imsave(output_path, adjusted_arr)
    print(f"Saved: {output_path}")

print("All adjusted images saved successfully!")

### SAM Cloud Segmentation with Edge/Border Tile Filtering

Now, we run the Segment Anything Model (SAM) Hybrid Prompting pipeline on our loaded imagery. To ensure accuracy and avoid skewed metrics (such as the ~82.8% cloud coverage error calculated on empty boundary black tiles), we first filter our tiles using the `is_not_black_tile` helper function.

In [ ]:
import urllib.request
import os

# Install segment-anything
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# Download SAM ViT-B model checkpoint
checkpoint_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
checkpoint_path = "sam_vit_b_01ec64.pth"

if not os.path.exists(checkpoint_path):
    print("Downloading SAM ViT-B model checkpoint...")
    urllib.request.urlretrieve(checkpoint_url, checkpoint_path)
    print("Download completed!")
else:
    print("Model checkpoint already exists.")

In [ ]:
from segment_anything import sam_model_registry, SamPredictor
import torch

# Load the model and initialize predictor
device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_b"](checkpoint=checkpoint_path)
sam.to(device=device)
predictor = SamPredictor(sam)
print(f"SAM model successfully loaded onto {device} and predictor initialized.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ensure predictor is available from the global scope
global predictor

# 1. Filter out the valid tiles using our helper function
valid_images = [(name, arr) for name, arr in images if arr is not None and is_not_black_tile(arr)]

processed_hybrid_results = []
whiteness_threshold = 175

print(f"Processing {len(valid_images)} non-black tiles with SAM hybrid prompting pipeline...")

# 2. Run SAM pipeline on a subset of valid tiles to save execution time
for i, (name, tile_image) in enumerate(valid_images[:5]):
    print(f"Processing tile [{i+1}/5]: {name}...")

    # Classical Whiteness Heuristic: Find bright regions in the valid tile
    gray_img = np.mean(tile_image, axis=-1)
    candidate_pixels = np.argwhere(gray_img > whiteness_threshold)

    if len(candidate_pixels) > 0:
        # Subsample to select up to 10 spaced-out prompt points
        step_size = max(1, len(candidate_pixels) // 10)
        selected_coords = candidate_pixels[::step_size][:10]

        input_points = np.array([[col, row] for row, col in selected_coords])
        input_labels = np.ones(len(input_points))

        # Predict refined cloud mask with SAM
        predictor.set_image(tile_image)
        masks, scores, logits = predictor.predict(
            point_coords=input_points,
            point_labels=input_labels,
            multimask_output=True
        )

        # Select the best scoring mask
        best_idx = np.argmax(scores)
        refined_mask = masks[best_idx]
        best_score = scores[best_idx]

        # Calculate cloud coverage percentage
        total_pixels = refined_mask.size
        cloud_pixels = np.sum(refined_mask)
        cloud_percentage = (cloud_pixels / total_pixels) * 100

        processed_hybrid_results.append({
            'name': name,
            'image': tile_image,
            'mask': refined_mask,
            'score': best_score,
            'cloud_percentage': cloud_percentage,
            'prompts': input_points
        })
        print(f"Successfully processed {name}! Cloud Coverage: {cloud_percentage:.2f}%, SAM Score: {best_score:.2f}")
    else:
        print(f"No bright cloud candidates found in {name}. Skipping.")

# 3. Visualize one of the processed valid results
if processed_hybrid_results:
    res = processed_hybrid_results[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(res['image'])
    axes[0].scatter(res['prompts'][:, 0], res['prompts'][:, 1], color='red', marker='*', s=100, label='Prompts')
    axes[0].set_title(f"Original Tile: {res['name'][:20]}")
    axes[0].legend()
    axes[0].axis('off')

    axes[1].imshow(res['image'])
    mask_visual = np.zeros_like(res['image'])
    mask_visual[res['mask']] = [0, 255, 0]  # Green overlay
    axes[1].imshow(mask_visual, alpha=0.5)
    axes[1].set_title(f"SAM Mask (Coverage: {res['cloud_percentage']:.2f}%)")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Calculate the cloud coverage percentage across all 23 valid tiles
all_cloud_percentages = []
whiteness_threshold = 175

print(f"Processing all {len(valid_images)} valid, non-black tiles to calculate mean cloud coverage...")

for i, (name, tile_image) in enumerate(valid_images):
    # Classical Whiteness Heuristic to act as prompt points
    gray_img = np.mean(tile_image, axis=-1)
    candidate_pixels = np.argwhere(gray_img > whiteness_threshold)

    if len(candidate_pixels) > 0:
        # Subsample up to 10 points
        step_size = max(1, len(candidate_pixels) // 10)
        selected_coords = candidate_pixels[::step_size][:10]
        input_points = np.array([[col, row] for row, col in selected_coords])
        input_labels = np.ones(len(input_points))

        # SAM Prediction
        predictor.set_image(tile_image)
        masks, scores, logits = predictor.predict(
            point_coords=input_points,
            point_labels=input_labels,
            multimask_output=True
        )

        best_idx = np.argmax(scores)
        refined_mask = masks[best_idx]

        # Calculate coverage percentage
        tile_cloud_percentage = (np.sum(refined_mask) / refined_mask.size) * 100
        all_cloud_percentages.append(tile_cloud_percentage)
    else:
        # No clouds detected
        all_cloud_percentages.append(0.0)

mean_cloud_coverage = np.mean(all_cloud_percentages)
print("\n=== FINAL CLOUD COVERAGE REPORT ===")
print(f"Total Tiles Evaluated: {len(all_cloud_percentages)}")
print(f"Mean Cloud Coverage  : {mean_cloud_coverage:.2f}%")

### Threshold Filtering
Filtering is handled inline during our main hybrid prediction pipeline using custom target parameters.

In [ ]:
!pip install -q pypdf

In [ ]:
import os

# List all files in the current directory
content_files = os.listdir('/content')
print("Files in /content:")
for f in content_files:
    print(f"- {f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np

# Let's extract the IoU values mentioned in the PDF for comparison:
# From Page 3 of the PDF:
# | Tile Type | Whiteness | Dark Channel | SAM-B (Everything) | SAM-H (Everything) | Hybrid (SAM-B) | Hybrid (SAM-H) |
# | 80% cloud |   0.84    |     0.87     |       0.15         |       0.84         |      0.81      |      0.84      |
# | 60% cloud |   0.61    |     0.64     |       0.60         |       0.65         |      0.61      |      0.68      |
# | 38% cloud |   0.74    |     0.77     |       0.51         |       0.75         |      0.82      |      0.84      |
# | 11% cloud |   0.10    |     0.14     |       0.00         |       0.00         |      0.17      |      0.19      |
#
# Overall small SAM (SAM-B) in everything mode collapses to a low average of 0.320 because of false alarms and missing faint segments.

print("=== PERFORMANCE COMPARISON: OUR RUN vs. PDF METRICS ===\n")

# Let's evaluate our segmentation on the first image as a sample
if 'example_adjusted_image' in globals() and 'cloud_like_masks' in globals() and len(cloud_like_masks) > 0:
    # Create a combined mask from our filtered cloud-like segments
    our_cloud_mask = np.zeros(example_adjusted_image.shape[:2], dtype=bool)
    for mask_data in cloud_like_masks:
        our_cloud_mask = np.logical_or(our_cloud_mask, mask_data['segmentation'])

    # Generate a simple proxy ground truth using a classic whiteness threshold (>150 intensity in the raw image)
    # Since our adjusted image has B=-100 and C=1.5, we'll use the raw image 'example_image_data'
    if 'example_image_data' in globals() and example_image_data is not None:
        raw_gray = np.mean(example_image_data, axis=-1)
        proxy_ground_truth = raw_gray > 150

        # Compute Intersection over Union (IoU) against this proxy
        intersection = np.logical_and(our_cloud_mask, proxy_ground_truth).sum()
        union = np.logical_or(our_cloud_mask, proxy_ground_truth).sum()
        our_iou = intersection / union if union > 0 else 0.0

        print(f"Our calculated IoU on test tile (against whiteness proxy): {our_iou:.3f}")
    else:
        print("Raw image not found to compute reference proxy IoU.")
else:
    print("No cloud masks found in session to evaluate.")

print("\nPDF Key Reference Metrics:")
print("- Small SAM (ViT-B) 'Everything Mode' Average IoU: ~0.320")
print("- Best Hybrid Zero-Shot (Classical Prompt -> SAM) IoU range: 0.59 to 0.84 (depending on cloud cover)")
print("- Standard Whiteness Rule IoU range: 0.10 (thin clouds) to 0.84 (dense clouds)")


In [ ]:
from segment_anything import SamPredictor
import numpy as np
import matplotlib.pyplot as plt

# Initialize SAM predictor
predictor = SamPredictor(sam)

# Prepare our test image
if 'example_adjusted_image' in globals() and example_adjusted_image is not None:
    img_to_segment = example_adjusted_image

    # 1. Apply Classical Rule: Find very bright regions to act as prompts
    # We'll calculate the mean intensity across channels
    gray_img = np.mean(img_to_segment, axis=-1)
    whiteness_threshold = 180
    candidate_pixels = np.argwhere(gray_img > whiteness_threshold)

    if len(candidate_pixels) > 0:
        # Subsample coordinates to avoid sending hundreds of redundant prompts
        # We select up to 10 spaced-out points as foreground prompts
        step_size = max(1, len(candidate_pixels) // 10)
        selected_coords = candidate_pixels[::step_size][:10]

        # SAM expects coordinates in (X, Y) format (column, row)
        input_points = np.array([[col, row] for row, col in selected_coords])
        input_labels = np.ones(len(input_points))  # 1 = Foreground Point

        # 2. Provide the image and classical prompts to SAM
        predictor.set_image(img_to_segment)
        masks, scores, logits = predictor.predict(
            point_coords=input_points,
            point_labels=input_labels,
            multimask_output=True
        )

        # Select the best scoring mask
        best_idx = np.argmax(scores)
        refined_mask = masks[best_idx]
        best_score = scores[best_idx]

        # 3. Visualize the Hybrid refinement result
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Original with prompts
        axes[0].imshow(img_to_segment)
        axes[0].scatter(input_points[:, 0], input_points[:, 1], color='red', marker='*', s=100, label='Prompts')
        axes[0].set_title("1. Classical Prompts (Whiteness)")
        axes[0].axis("off")
        axes[0].legend()

        # Classical rule fallback mask
        axes[1].imshow(gray_img > whiteness_threshold, cmap='gray')
        axes[1].set_title("2. Raw Whiteness Mask")
        axes[1].axis("off")

        # SAM Hybrid refined output
        axes[2].imshow(img_to_segment)
        mask_visual = np.zeros_like(img_to_segment)
        mask_visual[refined_mask] = [0, 255, 0] # Green overlay
        axes[2].imshow(mask_visual, alpha=0.5)
        axes[2].set_title(f"3. SAM Hybrid Refinement (Score: {best_score:.2f})")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No candidate pixels found exceeding the whiteness threshold.")
else:
    print("Example image not found in session.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Let's pick a different image index (e.g., the 3rd image, index 2) that is valid
selected_tile_idx = 2
tile_name, _, different_tile_image = adjusted_images[selected_tile_idx]

if different_tile_image is not None:
    print(f"Evaluating Hybrid Prompting on: {tile_name} (Index {selected_tile_idx})")

    # 1. Classical Rule: Find bright regions in this new tile
    gray_img = np.mean(different_tile_image, axis=-1)
    whiteness_threshold = 175  # Slightly adjusted threshold for contrast
    candidate_pixels = np.argwhere(gray_img > whiteness_threshold)

    if len(candidate_pixels) > 0:
        # Subsample to select 10 spaced-out prompt points
        step_size = max(1, len(candidate_pixels) // 10)
        selected_coords = candidate_pixels[::step_size][:10]

        input_points = np.array([[col, row] for row, col in selected_coords])
        input_labels = np.ones(len(input_points))  # 1 = Foreground Point

        # 2. Feed the new image and prompts to SAM
        predictor.set_image(different_tile_image)
        masks, scores, logits = predictor.predict(
            point_coords=input_points,
            point_labels=input_labels,
            multimask_output=True
        )

        # Get the highest scoring mask output
        best_idx = np.argmax(scores)
        refined_mask = masks[best_idx]
        best_score = scores[best_idx]

        # 3. Visualize results
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        # Original Image with the selected point prompts
        axes[0].imshow(different_tile_image)
        axes[0].scatter(input_points[:, 0], input_points[:, 1], color='red', marker='*', s=100, label='Prompts')
        axes[0].set_title("1. New Tile with Prompts")
        axes[0].axis("off")
        axes[0].legend()

        # Classical brightness mask alone
        axes[1].imshow(gray_img > whiteness_threshold, cmap='gray')
        axes[1].set_title("2. Raw Whiteness Mask")
        axes[1].axis("off")

        # SAM Hybrid prediction
        axes[2].imshow(different_tile_image)
        mask_visual = np.zeros_like(different_tile_image)
        mask_visual[refined_mask] = [0, 255, 0] # Green overlay
        axes[2].imshow(mask_visual, alpha=0.5)
        axes[2].set_title(f"3. SAM Refinement (Score: {best_score:.2f})")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No bright candidate pixels found to prompt SAM on this tile.")
else:
    print(f"Tile at index {selected_tile_idx} was invalid or missing.")

In [ ]:
# Calculate the cloud coverage percentage
total_pixels = refined_mask.size
cloud_pixels = np.sum(refined_mask)
cloud_percentage = (cloud_pixels / total_pixels) * 100

print(f"=== CLOUD COVERAGE BASELINE METRIC ===")
print(f"Total Image Resolution: {refined_mask.shape[1]}x{refined_mask.shape[0]} ({total_pixels} pixels)")
print(f"Detected Cloud Pixels : {cloud_pixels} pixels")
print(f"Cloud Coverage         : {cloud_percentage:.2f}%")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Run evaluation on all 23 valid tiles first to populate 'all_cloud_percentages'
all_cloud_percentages = []
whiteness_threshold = 175

print(f"Processing all {len(valid_images)} valid tiles with SAM to gather distribution data...")
for i, (name, tile_image) in enumerate(valid_images):
    gray_img = np.mean(tile_image, axis=-1)
    candidate_pixels = np.argwhere(gray_img > whiteness_threshold)

    if len(candidate_pixels) > 0:
        step_size = max(1, len(candidate_pixels) // 10)
        selected_coords = candidate_pixels[::step_size][:10]
        input_points = np.array([[col, row] for row, col in selected_coords])
        input_labels = np.ones(len(input_points))

        predictor.set_image(tile_image)
        masks, scores, logits = predictor.predict(
            point_coords=input_points,
            point_labels=input_labels,
            multimask_output=True
        )

        best_idx = np.argmax(scores)
        refined_mask = masks[best_idx]
        tile_cloud_percentage = (np.sum(refined_mask) / refined_mask.size) * 100
        all_cloud_percentages.append(tile_cloud_percentage)
    else:
        all_cloud_percentages.append(0.0)

# 2. Plot the histogram
plt.figure(figsize=(10, 6))
sns.histplot(all_cloud_percentages, bins=10, kde=True, color='skyblue', edgecolor='black')

mean_val = np.mean(all_cloud_percentages)
plt.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean Cloud Coverage: {mean_val:.2f}%')

plt.title('Distribution of Cloud Coverage Across Evaluated Tiles', fontsize=14, fontweight='bold')
plt.xlabel('Cloud Coverage (%)', fontsize=12)
plt.ylabel('Number of Tiles', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)

plt.tight_layout()
plt.show()